In [1]:
import os
import numpy as np
from scipy.stats import zscore
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import check_cv
from himalaya.kernel_ridge import KernelRidgeCV
from himalaya.backend import set_backend
from voxelwise_tutorials.delayer import Delayer
from voxelwise_tutorials.utils import zscore_runs, generate_leave_one_run_out
from voxelwise_tutorials.io import load_hdf5_array

/Users/shiochiba/miniconda3/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (None) doesn't match a supported version!
  warnings.warn(


In [ ]:
# Update these to point to data
DATA_DIR = '/Users/shiochiba/Documents/Projects/gaze_corrected_semantics/folder'
subject = "S01"  # change to subject ID

In [2]:
# Load fMRI responses
file_name = os.path.join(DATA_DIR, 'responses', f'{subject}_responses.hdf')

Y_train = load_hdf5_array(file_name, key="Y_train")
Y_test = load_hdf5_array(file_name, key="Y_test")

print("Y_train shape (n_samples_train, n_voxels):", Y_train.shape)
print("Y_test shape (n_repeats, n_samples_test, n_voxels):", Y_test.shape)

NameError: name 'DATA_DIR' is not defined

In [ ]:
# Preprocess fMRI responses

# Load run onset indices (marks where each run starts)
run_onsets = load_hdf5_array(file_name, key="run_onsets")

# Z-score each run independently to normalize within runs
Y_train = zscore_runs(Y_train, run_onsets)
Y_test = zscore(Y_test, axis=1)

# Average test repeats to get the repeatable signal
Y_test = Y_test.mean(0)
Y_test = zscore(Y_test, axis=0)

# Fill any NaN values with zeros
Y_train = np.nan_to_num(Y_train)
Y_test = np.nan_to_num(Y_test)

print("Preprocessed Y_train:", Y_train.shape)
print("Preprocessed Y_test:", Y_test.shape)

In [ ]:
# Compute explainable variance 

# Reload raw Y_test repeats to compute explainable variance
file_name = os.path.join(DATA_DIR, 'responses', f'{subject}_responses.hdf')
Y_test_repeats = load_hdf5_array(file_name, key="Y_test")

ev = explainable_variance(Y_test_repeats)
print("Max explainable variance:", ev.max().round(3))
print("Mean explainable variance:", ev.mean().round(3))

# Plot distribution across voxels
plt.hist(ev, bins=np.linspace(0, 1, 100), log=True, histtype='step')
plt.xlabel("Explainable variance")
plt.ylabel("Number of voxels")
plt.title("Histogram of explainable variance")
plt.grid('on')
plt.show()

In [ ]:
# Load stimulus features

# Replace "wordnet" with your feature space name (semantic segmentation features)
feature_space = "wordnet"

file_name = os.path.join(DATA_DIR, 'features', f'{feature_space}.hdf')
X_train = load_hdf5_array(file_name, key="X_train")
X_test = load_hdf5_array(file_name, key="X_test")

# Load standard semantic features
file_name_standard = os.path.join(DATA_DIR, 'features', 'standard_semantics.hdf')
X_train_standard = load_hdf5_array(file_name_standard, key="X_train")
X_test_standard = load_hdf5_array(file_name_standard, key="X_test")

# Load gaze-corrected semantic features
file_name_gaze = os.path.join(DATA_DIR, 'features', 'gaze_corrected_semantics.hdf')
X_train_gaze = load_hdf5_array(file_name_gaze, key="X_train")
X_test_gaze = load_hdf5_array(file_name_gaze, key="X_test")

print("X_train shape (n_samples_train, n_features):", X_train.shape)
print("X_test shape (n_samples_test, n_features):", X_test.shape)

In [ ]:
# Set backend

# Uses GPU if available, falls back to CPU automatically
backend = set_backend("torch_cuda", on_error="warn")
print("Backend:", backend)

# Use float32 for faster GPU computation
X_train = X_train.astype("float32")
X_test = X_test.astype("float32")

# for gaze corrected semantics
X_train_standard = X_train_standard.astype("float32")
X_test_standard = X_test_standard.astype("float32")
X_train_gaze = X_train_gaze.astype("float32")
X_test_gaze = X_test_gaze.astype("float32")

In [ ]:
# Define cross-validation

# Leave-one-run-out: keeps each recording run intact
# Important for fMRI since consecutive time points are correlated
n_samples_train = X_train.shape[0]

file_name = os.path.join(DATA_DIR, 'responses', f'{subject}_responses.hdf')
run_onsets = load_hdf5_array(file_name, key="run_onsets")

cv = generate_leave_one_run_out(n_samples_train, run_onsets)
cv = check_cv(cv)

In [ ]:
# fit standard semantic model

# Center features but don't normalize std
# (features are one-hot encoded so relative scale is meaningful)
scaler = StandardScaler(with_mean=True, with_std=False)

# 4 delays x 2 seconds covers the ~8 second hemodynamic response peak
delayer = Delayer(delays=[1, 2, 3, 4])

alphas = np.logspace(1, 20, 20)

kernel_ridge_cv = KernelRidgeCV(
    alphas=alphas,
    cv=cv,
    solver_params=dict(
        n_targets_batch=500,
        n_alphas_batch=5,
        n_targets_batch_refit=100
    )
)

pipeline_standard = make_pipeline(scaler, delayer, kernel_ridge_cv)

print("Fitting standard semantic model...")
_ = pipeline_standard.fit(X_train_standard, Y_train)

scores_standard = pipeline_standard.score(X_test_standard, Y_test)
scores_standard = backend.to_numpy(scores_standard)

print("Standard model - Mean R²:", scores_standard.mean().round(4))
print("Standard model - Max R²:", scores_standard.max().round(4))

In [ ]:
# Fit the model
_ = pipeline.fit(X_train, Y_train)
print("Model fitting complete!")

In [ ]:
# Plot the model prediction accuracy
import matplotlib.pyplot as plt
from voxelwise_tutorials.viz import plot_flatmap_from_mapper

mapper_file = os.path.join(directory, "mappers", f"{subject}_mappers.hdf")
ax = plot_flatmap_from_mapper(scores, mapper_file, vmin=0, vmax=0.4)
plt.show()

In [ ]:
# Verify best alphas aren't all at the edge of the range
# If they are, extend the range in np.logspace above
best_alphas_standard = backend.to_numpy(pipeline_standard[-1].best_alphas_)

plt.hist(np.log10(best_alphas_standard), bins=20)
plt.xlabel("log10(best alpha)")
plt.ylabel("Number of voxels")
plt.title("Distribution of best hyperparameters - Standard model")
plt.show()

In [ ]:
# fit gaze corrected model

# Rebuild pipeline with fresh estimator for gaze-corrected features
kernel_ridge_cv_gaze = KernelRidgeCV(
    alphas=alphas,
    cv=cv,
    solver_params=dict(
        n_targets_batch=500,
        n_alphas_batch=5,
        n_targets_batch_refit=100
    )
)

pipeline_gaze = make_pipeline(
    StandardScaler(with_mean=True, with_std=False),
    Delayer(delays=[1, 2, 3, 4]),
    kernel_ridge_cv_gaze
)

print("Fitting gaze-corrected model...")
_ = pipeline_gaze.fit(X_train_gaze, Y_train)

scores_gaze = pipeline_gaze.score(X_test_gaze, Y_test)
scores_gaze = backend.to_numpy(scores_gaze)

print("Gaze-corrected model - Mean R²:", scores_gaze.mean().round(4))
print("Gaze-corrected model - Max R²:", scores_gaze.max().round(4))

In [ ]:
# Check hyperparameters for gaze-corrected model
best_alphas_gaze = backend.to_numpy(pipeline_gaze[-1].best_alphas_)

plt.hist(np.log10(best_alphas_gaze), bins=20)
plt.xlabel("log10(best alpha)")
plt.ylabel("Number of voxels")
plt.title("Distribution of best hyperparameters - Gaze-corrected model")
plt.show()

In [ ]:
# Banded ridge regression fits both feature spaces at once and learns
# how much each feature space contributes to each voxel independently.

# Concatenate both feature spaces for train and test
X_train_both = np.hstack([X_train_standard, X_train_gaze])
X_test_both = np.hstack([X_test_standard, X_test_gaze])

# Tell himalaya which columns belong to which feature space
# (used to assign separate regularization per feature space)
feature_names = ["standard", "gaze_corrected"]
n_features_list = [X_train_standard.shape[1], X_train_gaze.shape[1]]

from himalaya.kernel_ridge import MultipleKernelRidgeCV
from himalaya.kernel_ridge import Kernelizer, ColumnKernelizer

# Build a kernelizer for each feature space
kernelizers = [
    (name, Kernelizer(), slice(start, start + n_feat))
    for name, n_feat, start in zip(
        feature_names,
        n_features_list,
        np.concatenate([[0], np.cumsum(n_features_list)[:-1]])
    )
]
column_kernelizer = ColumnKernelizer(kernelizers)

# MultipleKernelRidgeCV fits a separate regularization weight per feature space per voxel
mk_ridge_cv = MultipleKernelRidgeCV(kernels="precomputed", cv=cv)

pipeline_banded = make_pipeline(
    StandardScaler(with_mean=True, with_std=False),
    Delayer(delays=[1, 2, 3, 4]),
    column_kernelizer,
    mk_ridge_cv
)

print("Fitting banded ridge model...")
_ = pipeline_banded.fit(X_train_both, Y_train)

scores_banded = pipeline_banded.score(X_test_both, Y_test)
scores_banded = backend.to_numpy(scores_banded)

print("Banded ridge model - Mean R²:", scores_banded.mean().round(4))
print("Banded ridge model - Max R²:", scores_banded.max().round(4))

In [ ]:
# Compare all 3 models
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, scores, title in zip(
    axes,
    [scores_standard, scores_gaze, scores_banded],
    ["Standard semantics", "Gaze-corrected semantics", "Banded ridge (both)"]
):
    ax.hist(scores, bins=50)
    ax.axvline(scores.mean(), color='red', linestyle='--',
               label=f'Mean: {scores.mean():.3f}')
    ax.set_xlabel("R² score")
    ax.set_ylabel("Number of voxels")
    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.show()

# Print summary comparison
print(f"{'Model':<30} {'Mean R²':>10} {'Max R²':>10}")
print("-" * 50)
print(f"{'Standard semantics':<30} {scores_standard.mean():>10.4f} {scores_standard.max():>10.4f}")
print(f"{'Gaze-corrected semantics':<30} {scores_gaze.mean():>10.4f} {scores_gaze.max():>10.4f}")
print(f"{'Banded ridge (both)':<30} {scores_banded.mean():>10.4f} {scores_banded.max():>10.4f}")

import numpy as np

results_dir = os.path.join(DATA_DIR, 'results')
os.makedirs(results_dir, exist_ok=True)

np.save(os.path.join(results_dir, f'{subject}_scores_standard.npy'), scores_standard)
np.save(os.path.join(results_dir, f'{subject}_scores_gaze.npy'), scores_gaze)
np.save(os.path.join(results_dir, f'{subject}_scores_banded.npy'), scores_banded)

print(f"Results saved to {results_dir}")